In [63]:
import pandas as pd
import numpy as np 
import plotly.express as px

In [64]:
df_day = pd.read_csv("../data/intermin/data_daily.csv")
df_hour = pd.read_csv("../data/intermin/data_hourly.csv")

## Actual Consumption

### Daily Trend

In [65]:
df_day = df_day.sort_values("date_from")
df_day["actual_consumption_ma7"] = df_day["actual_consumption"].rolling(window = 7).mean()

fig = px.line(df_day, 
              x = "date_from", 
              y = ["actual_consumption", "actual_consumption_ma7"],
              color_discrete_sequence=["#3D97CA", "#E69F00"],
              title = "Actual Consumption Daily Trend: 2019 - 2025")

fig.update_xaxes(title = "year")
fig.update_yaxes(title = "MWh/day")
fig.update_layout(
    legend=dict(
        orientation="h",      # horizontal
        yanchor="top",
        y=-0.2,               # below the plot area (negative = outside)
        xanchor="center",
        x=0.5,
        title_text="",        # remove the "variable" legend title
    ),
    title_x = 0.5
)
fig.show()

- **Observation**: The daily series swings between ≈ 0.87M (2024-03-31, Easter Sunday) and ≈ 1.68M MWh/day (2021-11-30). The 7-day average removes the weekly cycle and leaves a clear **annual wave**: highs in winter (up to ≈ 1.57M in Feb 2021), lows in summer (down to ≈ 1.10M in May 2023). The yearly maxima are lower in 2023 (≈ 1.57M) and 2024 (≈ 1.60M) than in 2019–2022 (≈ 1.65–1.68M).
- **Interpretation**: The wide spread of the raw line comes from the weekday/weekend cycle (see Weekday Effect) and from holidays; the 7-day average hides both, so it should not be used to judge single days. The lower winter peaks after 2022 are consistent with the 2022–2023 drop found later in the MSTL trend.
- **Implication**: Seasonality and the weekly cycle dominate the variance, so any trend or event comparison needs them removed first (see the by-month and STL sections).

### Mean consumption by month, per year

In [66]:
df_day["date_from"] = pd.to_datetime(df_day["date_from"])
df_day["year"] = df_day["date_from"].dt.year

monthly = (
    df_day.groupby(["year", "month"], as_index=False)["actual_consumption"].mean()
)
monthly["year"] = monthly["year"].astype(str)  # categorical, so each year gets a distinct color

fig = px.line(
    monthly,
    x = "month",
    y = "actual_consumption",
    color = "year",
    markers = True,
    title = "Mean daily actual consumption by month, per year",
    labels = {"actual_consumption": "MWh/day", "month": "Month", "year": "Year"},
    color_discrete_sequence = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00", "#CC79A7"],  # Okabe-Ito
)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_layout(
    legend = dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5, title_text=""),
    title_x = 0.5
)
fig.show()


- **Observation**: Winter peaks (~1.5M MWh/day), summer troughs (~1.2M). 2020 drops in Mar–May, 2022 drops in H2, 2023 is the lowest year overall.
- **Interpretation**: 2020 matches the first lockdown; 2022 matches the energy crisis (high prices, saving). Weather is not controlled here.
- **Implication**: Month is a strong feature; year-level shifts are not purely seasonal.


### Weekday Effect

In [67]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

df_day["date_from"] = pd.to_datetime(df_day["date_from"])
df_day["year"] = df_day["date_from"].dt.year

# 1. Remove level/season: ratio to centered 7-day mean
df_day["ratio_to_week"] = (
    df_day["actual_consumption"]
    / df_day["actual_consumption"].rolling(7, center=True).mean()
)

# 2. Day type: holidays on weekdays are their own group
def day_type(r):
    weekend = r["day_of_week_raw"] >= 5
    if r["is_holiday"]:
        return "Weekend holiday" if weekend else "Weekday holiday"
    return "Weekend" if weekend else "Weekday"

day_type_order = ["Weekday", "Weekday holiday", "Weekend", "Weekend holiday"]
df_day["day_type"] = pd.Categorical(
    df_day.apply(day_type, axis=1), categories=day_type_order, ordered=True
)


In [68]:
# (a) 4 day types, pooled across years: box plot
fig = px.box(df_day.dropna(subset=["ratio_to_week"]),
             x="day_type", y="ratio_to_week", points="all",
             title="Day type effect (ratio to centered 7-day mean)")
fig.update_traces(marker=dict(size=3, opacity=0.5), jitter=0.4)
fig.update_layout(title_x = 0.5)
fig.show()

In [69]:
# (b) Mon-Sun detail by year, normal days only
normal = df_day[df_day["day_type"].isin(["Weekday", "Weekend"])].copy()
normal["day_of_week"] = pd.Categorical(normal["day_of_week"], categories=order, ordered=True)
dow = (normal.groupby(["year", "day_of_week"], observed=True, as_index=False)["ratio_to_week"].mean()
             .sort_values(["year", "day_of_week"]))
dow["year"] = dow["year"].astype(str)
fig = px.line(dow, x="day_of_week", y="ratio_to_week", color="year", markers=True,
              title="Weekday effect, holidays excluded")
fig.update_layout(title_x = 0.5)
fig.show()

##
- **Observation**: Mon–Fri ≈ 1.04–1.07x of the weekly mean, Saturday ≈ 0.90x, Sunday ≈ 0.84x. The pattern is almost identical across 2019–2024.
- **Interpretation**: Economic is less active during weekends (people do not work). Monday and Friday have slightly lower consumption, possibly due to people working remotely on that two days.
- **Implication**: day_type (4 classes) is a useful feature; weekend holidays are NOT lower than regular weekends.

### Intra-day Pattern by Season

In [70]:
df_hour["date_from"] = pd.to_datetime(df_hour["date_from"])

# 1. Drop DST switch days (23 or 25 hours)
day = df_hour["date_from"].dt.floor("D")
n_hours = df_hour.groupby(day)["date_from"].transform("size")
h = df_hour[n_hours == 24].copy()

# 2. Features
h["hour"] = h["date_from"].dt.hour

day_type_order = ["Weekday", "Weekday holiday", "Weekend", "Weekend holiday"]
h["day_type"] = pd.Categorical(h.apply(day_type, axis=1), categories=day_type_order, ordered=True)

season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer",
              9: "Autumn", 10: "Autumn", 11: "Autumn"}
season_order = ["Winter", "Spring", "Summer", "Autumn"]

h["season"] = h["month"].map(season_map)

# remove the month filter (h = h[h["month"].isin(...)]) from the earlier code
profile = (
    h.groupby(["season", "day_type", "hour"], observed=True, as_index=False)["actual_consumption"].mean()
)

fig = px.line(
    profile, x="hour", y="actual_consumption", color="day_type",
    facet_col="season", category_orders={"season": season_order, "day_type": day_type_order},
    title="Average 24h load profile",
    labels={"actual_consumption": "MWh/h", "hour": "Hour of day"},
)
fig.update_xaxes(tickmode="linear", dtick=4)
fig.update_layout(title_x=0.5)
fig.show()



- **Observation** (hourly means, DST switch days excluded):
  - Winter weekday: night low ≈ 49k MWh/h (02:00), then a plateau of ≈ 70k with **two peaks at 11:00 and 18:00**, and only a slight dip in between (≈ 67k at 15:00).
  - Summer weekday: night low ≈ 42k (02:00), **a single peak at 11:00 (≈ 64k)**, then a gradual decline through the evening. No evening peak.
  - Weekend: the whole curve is lower and flatter. The morning ramp-up (06:00–09:00) is much weaker. In winter the weekend maximum is in the evening (18:00, ≈ 61k).
  - Weekday holidays are lower than regular weekends in every hour (winter ≈ 38–52k vs ≈ 46–61k; summer ≈ 35–49k vs ≈ 38–51k).
- **Interpretation**: The 06:00–09:00 ramp on weekdays reflects the start of work and industrial activity, which is absent on weekends. The winter evening peak is most likely lighting and heating when people are at home; it is missing in summer because it is still light. Weekday holidays being below weekends in absolute terms contrasts with the ratio view above (≈ 0.89 vs ≈ 0.87), because the ratio is relative to a weekly mean that the holiday itself pulls down. Holidays are also concentrated in winter and spring (Christmas, New Year, Easter), so the season mix is not identical to regular weekends.
- **Implication**: The hour-of-day shape depends on both day type and season, so an hourly model needs `hour × day_type` and `hour × season` interactions (or separate models). A daily model can ignore this. Spring and autumn are plotted but not discussed here.

### STL Decomposition - Daily Data

In [71]:
from statsmodels.tsa.seasonal import MSTL

s = df_day.set_index("date_from")["actual_consumption"].asfreq("D")
res = MSTL(s, periods=(7, 365)).fit()
# res.seasonal has columns seasonal_7 and seasonal_365

In [72]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

comp = pd.DataFrame({
    "observed": res.observed, "trend": res.trend,
    "seasonal_7": res.seasonal["seasonal_7"], "seasonal_365": res.seasonal["seasonal_365"],
    "resid": res.resid,
})

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=list(comp.columns[:1]) + ["trend", "weekly seasonal", "yearly seasonal"])
for i, c in enumerate(["observed", "trend", "seasonal_7", "seasonal_365"], start=1):
    fig.add_trace(go.Scatter(x=comp.index, y=comp[c], mode="lines", showlegend=False), row=i, col=1)
fig.update_layout(height=700, title_text="MSTL decomposition", title_x=0.5)
fig.show()



# Residual: where are the anomalies?
fig = px.line(comp, y="resid", title="Residual")
fig.update_layout(height=700, title_x=0.5)
fig.show()


- **Observation**:
  - **Trend** (MSTL, periods = 7 and 365): ≈ 1.38M MWh/day (Jan 2019) → ≈ 1.33M (mid 2020) → back to a peak of ≈ 1.38M (Aug 2021) → a low of ≈ 1.25M (Apr 2023, about −9% from the 2021 peak) → ≈ 1.28M (end of 2024). The decline is not monotonic; most of it is one drop in 2022–2023.
  - **Weekly seasonal**: ≈ +100k on weekdays and ≈ −200k on weekends, stable over the six years (too dense to read at full range; zoom into a few weeks).
  - **Yearly seasonal**: winter high (≈ +200k), summer low, and a sharp drop around Christmas and New Year (≈ −200k to −300k).
  - **Residual**: ~39 days exceed ±3σ. Almost all are calendar events: Good Friday, Easter Monday, Ascension Day and Whit Monday (−130k to −230k) in every year. Weekend holidays (e.g. 2021-12-25/26) show **positive** residuals (+130k to +160k). The one unexplained cluster is **2022-04-01 to 2022-04-10** (+130k to +180k, 1.57M on 04-01).
- **Interpretation**: Christmas and New Year have fixed dates, so the yearly seasonal absorbs them. Easter-related holidays move every year, so they stay in the residual. Weekend holidays are positive because the model subtracts both the weekend effect and the holiday effect, but the two do not add up. The 2020 dip and the 2022–2023 drop are consistent with the first lockdown and the energy crisis, but this is not tested against temperature or price data. The MSTL trend is very smooth, so it blurs the exact timing of events.
- **Implication**: Keep `is_holiday`: it explains the large negative residuals on weekday holidays. Add the 4-class `day_type` (weekend × holiday) because weekend holidays do not behave like "weekend + holiday". Easter-specific features (days to Easter, Holy Saturday / Easter Sunday flags) and bridge-day flags are not justified yet: Holy Saturday and Easter Sunday are close to regular weekends (≈ 0.92 vs 0.90 and ≈ 0.83 vs 0.85 of the weekly mean), and only one bridge day (2021-05-14) shows up. Revisit only if model residuals still show the pattern. The 2022-04 cluster needs a check against the hourly data and a temperature series before deciding whether to keep or flag those days. The first and last half-year of the 365-day seasonal rest on few cycles and should not be over-interpreted.
- **TODO**: check 2022-04 hourly data; compare with temperature (e.g. DWD); FFT / power spectrum to confirm the 7- and 365-day periods.